In [ ]:
import os
import json
import glob
import math
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
import statsmodels.formula.api as smf


# ----------------------------
# Config
# ----------------------------
RESULT_DIR = "./result"
OUT_DIR = "./analysis_out/ch5_3"
FIG_DIR = os.path.join(OUT_DIR, "figures")
TAB_DIR = os.path.join(OUT_DIR, "tables")

EVENT_ORDER = ["Continue", "SizeChange", "Merge", "Split"]
W_ORDER = [1, 2, 5, 10, 20, 50]

K_COMM = 3
HALIAS_NORM_DEN = math.log2(K_COMM)

N_BINS = 10

# RT trimming (avoid extreme long outliers wrecking Gaussian models)
# You can change these if needed
RT_MIN_MS = 200
RT_MAX_MS = 60000  # 60s cap


# ----------------------------
# Utilities
# ----------------------------
def ensure_dirs():
    os.makedirs(FIG_DIR, exist_ok=True)
    os.makedirs(TAB_DIR, exist_ok=True)

def save_csv(df: pd.DataFrame, name: str):
    path = os.path.join(TAB_DIR, name)
    df.to_csv(path, index=False)
    print(f"[saved] {path}")

def save_txt(text: str, name: str):
    path = os.path.join(TAB_DIR, name)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"[saved] {path}")

# ----------------------------
# Load
# ----------------------------
def load_all_results(result_dir: str) -> pd.DataFrame:
    paths = sorted(glob.glob(os.path.join(result_dir, "*.json")))
    if not paths:
        raise FileNotFoundError(f"No JSON files found in {result_dir}")

    rows = []
    for p in paths:
        with open(p, "r", encoding="utf-8") as f:
            data = json.load(f)
        if not isinstance(data, list):
            raise ValueError(f"Result file must contain a list of trials: {p}")

        for t in data:
            gt = t.get("ground_truth")
            gt = "Continue" if gt == "Stable" else gt

            rows.append({
                "source_file": os.path.basename(p),
                "subject_id": t.get("subject_id"),
                "trial_index": t.get("trial_index"),
                "W": t.get("W"),
                "ground_truth": gt,
                "correct": t.get("correct"),
                "duration_ms": t.get("duration_ms"),
                "confidence": t.get("confidence"),
                "space_toggle_count": t.get("space_toggle_count"),
                "mouse_entropy": t.get("mouse_entropy"),
                "h_alias": t.get("h_alias"),
            })

    df = pd.DataFrame(rows)

    # numeric conversions
    for c in ["trial_index", "W", "correct", "duration_ms", "confidence",
              "space_toggle_count", "mouse_entropy", "h_alias"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df["subject_id"] = df["subject_id"].astype(str)
    df["ground_truth"] = df["ground_truth"].astype(str)

    # basic validity
    df = df.dropna(subset=["subject_id", "W", "ground_truth", "h_alias",
                           "duration_ms", "space_toggle_count", "mouse_entropy", "confidence"]).copy()

    df["halias_norm"] = df["h_alias"] / HALIAS_NORM_DEN

    # RT trimming + transform
    df["rt_ms_raw"] = df["duration_ms"].astype(float)
    df = df[(df["rt_ms_raw"] >= RT_MIN_MS) & (df["rt_ms_raw"] <= RT_MAX_MS)].copy()
    df["log_rt"] = np.log1p(df["rt_ms_raw"])

    # categorical order
    df["ground_truth"] = pd.Categorical(df["ground_truth"], categories=EVENT_ORDER, ordered=True)
    df["W"] = pd.Categorical(df["W"], categories=W_ORDER, ordered=True)

    return df


# ----------------------------
# Binned curves (mean ±95% CI)
# ----------------------------
def bin_summary(df: pd.DataFrame, x: str, y: str, n_bins: int, group: Optional[str] = None) -> pd.DataFrame:
    tmp = df[[x, y] + ([group] if group else [])].dropna().copy()

    def summarize_one(sub: pd.DataFrame) -> pd.DataFrame:
        try:
            sub["bin"] = pd.qcut(sub[x], q=n_bins, duplicates="drop")
        except Exception:
            sub["bin"] = pd.cut(sub[x], bins=n_bins)

        g = sub.groupby("bin", dropna=False)
        out = g.agg(
            n=(y, "count"),
            x_mean=(x, "mean"),
            mean=(y, "mean"),
            sd=(y, "std"),
        ).reset_index(drop=True)

        # 95% CI for mean using normal approx: mean ± 1.96 * sd/sqrt(n)
        n = out["n"].astype(float).replace(0, np.nan)
        se = out["sd"].astype(float) / np.sqrt(n)
        out["ci95_lo"] = out["mean"] - 1.96 * se
        out["ci95_hi"] = out["mean"] + 1.96 * se
        return out.sort_values("x_mean")

    if group:
        frames = []
        for gname, sub in tmp.groupby(group):
            out = summarize_one(sub)
            out[group] = gname
            frames.append(out)
        return pd.concat(frames, ignore_index=True)
    else:
        return summarize_one(tmp)


def plot_binned_curve(bin_df: pd.DataFrame, ylabel: str, title: str, out_png: str):
    plt.figure()
    x = bin_df["x_mean"].values
    y = bin_df["mean"].values
    lo = bin_df["ci95_lo"].values
    hi = bin_df["ci95_hi"].values
    plt.plot(x, y, marker="o")
    plt.fill_between(x, lo, hi, alpha=0.2)
    plt.title(title)
    plt.xlabel(r"Normalized aliasing entropy $\tilde{H}_{alias}$")
    plt.ylabel(ylabel)
    plt.tight_layout()
    outpath = os.path.join(FIG_DIR, out_png)
    plt.savefig(outpath, dpi=220)
    plt.close()
    print(f"[saved] {outpath}")


def plot_binned_curve_by_event(bin_df: pd.DataFrame, ylabel: str, title: str, out_png: str):
    plt.figure()
    for ev in EVENT_ORDER:
        if ev not in bin_df["ground_truth"].unique():
            continue
        sub = bin_df[bin_df["ground_truth"] == ev]
        if sub.empty:
            continue
        plt.plot(sub["x_mean"].values, sub["mean"].values, marker="o", label=ev)
    plt.title(title)
    plt.xlabel(r"Normalized aliasing entropy $\tilde{H}_{alias}$")
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    outpath = os.path.join(FIG_DIR, out_png)
    plt.savefig(outpath, dpi=220)
    plt.close()
    print(f"[saved] {outpath}")


# ----------------------------
# GEE models (Gaussian for effort proxies)
# ----------------------------
def fit_gee_gaussian(formula: str, df: pd.DataFrame):
    return smf.gee(
        formula=formula,
        groups="subject_id",
        data=df,
        family=sm.families.Gaussian()
    ).fit()


def summarize_model(model, name: str) -> pd.DataFrame:
    params = model.params
    se = model.bse
    z = params / se

    # p-values using normal approx (no scipy needed)
    def norm_cdf(x: float) -> float:
        return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

    p = np.array([2.0 * (1.0 - norm_cdf(abs(float(zi)))) for zi in z])

    out = pd.DataFrame({
        "term": params.index,
        "beta": params.values,
        "se": se.values,
        "z": z.values,
        "p": p,
        "ci95_lo": params.values - 1.96 * se.values,
        "ci95_hi": params.values + 1.96 * se.values,
        "model": name
    })
    return out


def write_key_line(summary_df: pd.DataFrame, term: str, outname: str, note: str = ""):
    row = summary_df[summary_df["term"] == term]
    if row.empty:
        save_txt(f"Term not found: {term}\n", outname)
        return
    r = row.iloc[0]
    txt = (
        f"{note}\n" if note else ""
    ) + (
        f"Key effect ({term}):\n"
        f"beta = {r['beta']:.6f}\n"
        f"SE   = {r['se']:.6f}\n"
        f"z    = {r['z']:.3f}\n"
        f"p    = {r['p']:.4g}\n"
        f"95% CI [{r['ci95_lo']:.6f}, {r['ci95_hi']:.6f}]\n"
    )
    save_txt(txt, outname)


# ----------------------------
# Main pipeline for 5.3
# ----------------------------
def main():
    ensure_dirs()
    df = load_all_results(RESULT_DIR)

    # Overview for paper
    overview = (
        f"N_subjects = {df['subject_id'].nunique()}\n"
        f"N_trials (after RT trimming) = {len(df)}\n"
        f"RT trimming: [{RT_MIN_MS}, {RT_MAX_MS}] ms\n"
        f"halias_norm range = [{df['halias_norm'].min():.3f}, {df['halias_norm'].max():.3f}]\n"
    )
    save_txt(overview, "table_5_3_overview.txt")

    # ----------------------------
    # Descriptive curves (overall)
    # ----------------------------
    # RT (raw)
    bins_rt = bin_summary(df, x="halias_norm", y="rt_ms_raw", n_bins=N_BINS)
    save_csv(bins_rt, "table_5_3_rt_binned.csv")
    plot_binned_curve(
        bins_rt,
        ylabel="Response time (ms)",
        title=r"Response time vs $\tilde{H}_{alias}$ (binned mean $\pm$95\% CI)",
        out_png="fig_5_3_rt_vs_halias_binned.png"
    )

    # RT (log)
    bins_lrt = bin_summary(df, x="halias_norm", y="log_rt", n_bins=N_BINS)
    save_csv(bins_lrt, "table_5_3_logrt_binned.csv")
    plot_binned_curve(
        bins_lrt,
        ylabel=r"$\log(1+\mathrm{RT})$",
        title=r"Log response time vs $\tilde{H}_{alias}$ (binned mean $\pm$95\% CI)",
        out_png="fig_5_3_logrt_vs_halias_binned.png"
    )

    # Space toggles
    bins_tog = bin_summary(df, x="halias_norm", y="space_toggle_count", n_bins=N_BINS)
    save_csv(bins_tog, "table_5_3_toggles_binned.csv")
    plot_binned_curve(
        bins_tog,
        ylabel="Space toggles",
        title=r"Space toggles vs $\tilde{H}_{alias}$ (binned mean $\pm$95\% CI)",
        out_png="fig_5_3_toggles_vs_halias_binned.png"
    )

    # Mouse entropy
    bins_me = bin_summary(df, x="halias_norm", y="mouse_entropy", n_bins=N_BINS)
    save_csv(bins_me, "table_5_3_mouse_entropy_binned.csv")
    plot_binned_curve(
        bins_me,
        ylabel="Mouse entropy",
        title=r"Mouse entropy vs $\tilde{H}_{alias}$ (binned mean $\pm$95\% CI)",
        out_png="fig_5_3_mouse_entropy_vs_halias_binned.png"
    )

    # Confidence
    bins_conf = bin_summary(df, x="halias_norm", y="confidence", n_bins=N_BINS)
    save_csv(bins_conf, "table_5_3_confidence_binned.csv")
    plot_binned_curve(
        bins_conf,
        ylabel="Confidence",
        title=r"Confidence vs $\tilde{H}_{alias}$ (binned mean $\pm$95\% CI)",
        out_png="fig_5_3_confidence_vs_halias_binned.png"
    )

    # (Optional) By-event RT(log) curves (useful for appendix or 5.4)
    bins_lrt_ev = bin_summary(df, x="halias_norm", y="log_rt", n_bins=max(6, N_BINS // 2), group="ground_truth")
    save_csv(bins_lrt_ev, "table_5_3_logrt_binned_by_event.csv")
    plot_binned_curve_by_event(
        bins_lrt_ev,
        ylabel=r"$\log(1+\mathrm{RT})$",
        title=r"Log response time vs $\tilde{H}_{alias}$ by event type",
        out_png="fig_5_3_logrt_vs_halias_binned_by_event.png"
    )

    # ----------------------------
    # Statistical models (GEE Gaussian)
    #   main: effort ~ halias_norm
    #   controls: + event + W
    #   robustness: exclude Continue
    # ----------------------------
    # RT (log)
    m_rt = fit_gee_gaussian("log_rt ~ halias_norm", df)
    m_rt_c = fit_gee_gaussian("log_rt ~ halias_norm + C(ground_truth) + C(W)", df)

    s_rt = summarize_model(m_rt, "GEE_logRT_main")
    s_rt_c = summarize_model(m_rt_c, "GEE_logRT_controls")
    save_csv(s_rt, "table_5_3_gee_logrt_main.csv")
    save_csv(s_rt_c, "table_5_3_gee_logrt_controls.csv")
    write_key_line(s_rt, "halias_norm", "table_5_3_key_logrt_main.txt")
    write_key_line(s_rt_c, "halias_norm", "table_5_3_key_logrt_controls.txt")

    # toggles
    m_t = fit_gee_gaussian("space_toggle_count ~ halias_norm", df)
    m_t_c = fit_gee_gaussian("space_toggle_count ~ halias_norm + C(ground_truth) + C(W)", df)
    s_t = summarize_model(m_t, "GEE_toggles_main")
    s_t_c = summarize_model(m_t_c, "GEE_toggles_controls")
    save_csv(s_t, "table_5_3_gee_toggles_main.csv")
    save_csv(s_t_c, "table_5_3_gee_toggles_controls.csv")
    write_key_line(s_t, "halias_norm", "table_5_3_key_toggles_main.txt")
    write_key_line(s_t_c, "halias_norm", "table_5_3_key_toggles_controls.txt")

    # mouse entropy
    m_me = fit_gee_gaussian("mouse_entropy ~ halias_norm", df)
    m_me_c = fit_gee_gaussian("mouse_entropy ~ halias_norm + C(ground_truth) + C(W)", df)
    s_me = summarize_model(m_me, "GEE_mouseEntropy_main")
    s_me_c = summarize_model(m_me_c, "GEE_mouseEntropy_controls")
    save_csv(s_me, "table_5_3_gee_mouse_entropy_main.csv")
    save_csv(s_me_c, "table_5_3_gee_mouse_entropy_controls.csv")
    write_key_line(s_me, "halias_norm", "table_5_3_key_mouse_entropy_main.txt")
    write_key_line(s_me_c, "halias_norm", "table_5_3_key_mouse_entropy_controls.txt")

    # confidence
    m_cf = fit_gee_gaussian("confidence ~ halias_norm", df)
    m_cf_c = fit_gee_gaussian("confidence ~ halias_norm + C(ground_truth) + C(W)", df)
    s_cf = summarize_model(m_cf, "GEE_confidence_main")
    s_cf_c = summarize_model(m_cf_c, "GEE_confidence_controls")
    save_csv(s_cf, "table_5_3_gee_confidence_main.csv")
    save_csv(s_cf_c, "table_5_3_gee_confidence_controls.csv")
    write_key_line(s_cf, "halias_norm", "table_5_3_key_confidence_main.txt")
    write_key_line(s_cf_c, "halias_norm", "table_5_3_key_confidence_controls.txt")

    # robustness: exclude Continue
    df_nc = df[df["ground_truth"] != "Continue"].copy()
    if len(df_nc) > 0:
        m_rt_nc = fit_gee_gaussian("log_rt ~ halias_norm", df_nc)
        s_rt_nc = summarize_model(m_rt_nc, "GEE_logRT_noContinue")
        save_csv(s_rt_nc, "table_5_3_gee_logrt_noContinue.csv")
        write_key_line(s_rt_nc, "halias_norm", "table_5_3_key_logrt_noContinue.txt", note="Exclude Continue trials")

    print("\n✅ 5.3 analysis complete.")
    print(f"Outputs in: {OUT_DIR}")


if __name__ == "__main__":
    main()